# 🫀 퀘스트 46 · Q4-I — **문제는 특징이 아니라 추정기다**: 능력·차원 통제

| | **MedKOS / `notebooks/quest46_q4i_capacity_control.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — **층① 표현**, 단 이번엔 **추정기**를 고친다 |
| 부모 런 | `quest46_q4h_rhythm_context`(`20260805T0726`) |
| 성격 | **Q4-H 의 판정을 다시 읽는다** — 단, 내 재독 자체를 **반증할 수 있게** 짓는다 |
| 예상 소요 | **25–35분** (Q4-H 690초의 3~4배 — 팔이 6개 + 영점 두 벌) |

## ★★★ Q4-H 의 판정을 다시 읽는다

Q4-H 는 「교과서적 특징이 오히려 해롭다(L3 ❌)」로 끝났다. 실측은 이랬다.

```
팔     차원   매크로 AUROC       Δ vs base
base    9      0.9420            —
comp   14      0.9375           −0.0044 [−0.0072, −0.0022]
ctx    20      0.9285           −0.0134 [−0.0201, −0.0075]
both   25      0.9278           −0.0142 [−0.0214, −0.0078]
영점(라벨 치환) **−0.0427**      문턱 = max(0, 영점 상단) = **+0.0000**
```

### 여기서 내가 놓친 것 — **영점이 −0.0427 인데 문턱을 0 으로 잘랐다**

라벨을 치환하면(=신호 0) 같은 특징 추가가 매크로 AUROC 를 **−0.0427** 떨어뜨린다.
즉 **차원을 16개 늘리는 행위 자체의 값이 −0.0427** 이다. 관측된 −0.0142 는 그 바닥보다
**+0.0285 위**다.

| 읽는 법 | 문턱 | `both` −0.0142 | 뜻 |
|---|---|---|---|
| **배포 관문**(절대) | `max(0, 영점상단)` = 0 | ❌ | 지금 그대로 넣으면 **모델이 나빠진다** |
| **기전 관문**(영점 대비) | 영점상단 ≈ −0.03 | ✅ | 추가 특징은 **실제 신호를 갖는다** |

Q4-H 는 **배포 관문만 보고**했다. 그래서 「특징이 해롭다」로 읽혔지만, 잘린 문턱은
**기전 판정을 아예 못 내게** 한다.

### ⚠️ 그런데 — **내 재독을 반증할 수 있는 대안설명을 찾았다**(R39 ①)

영점을 **교정된 점수**로 쟀다는 게 문제다. Platt 교정기는 **실제 DEV 라벨**로 적합된다.
학습 라벨을 치환하면 모델은 잡음 방향을 향하는데, 그때 교정기 기울기가 **음수로 나와
부호를 되살린다**(AUROC → 1−AUROC). 이 되살림은 저차원에서 더 잘 되므로,
**차원이 커질수록 영점이 내려간 것처럼 보인다** — 능력 비용이 없어도.

```
합성 확인(스모크) — 영점(raw) **+0.0066**  vs  영점(교정) **−0.0295**   차 −0.0361
```

**즉 음수 영점의 거의 전부가 교정 artifact 일 수 있다.** 신호 조건에서는 기울기가
양수라 교정이 AUROC 를 **정확히 안 바꾼다**(스모크 실측 최대차 `0.00e+00`) — 그래서
**영점만 raw 로 다시 재면 사과 대 사과**가 된다. 이 런은 **둘 다 재서 병기**한다.
내 재독이 틀렸으면 이 자리에서 죽는다.

**⇒ 그래서 판정을 「영점 해석」에 걸지 않는다. 주 관문은 차원을 맞춘 `shuf` 대조다.**

### 그 진단을 뒷받침하는 두 번째 증거 — **차원에 단조**

```
차원  9 → 14 → 20 → 25
AUROC 0.9420 → 0.9375 → 0.9285 → 0.9278
```

특징의 **내용**과 무관하게 **차원이 커질수록 정확히 단조로 나빠진다**. 내용이 원인이라면
`comp`(생리적으로 강한 판별자 5개)와 `ctx`(약한 11개)가 이런 깔끔한 차원 순서를 만들 이유가
없다. **비용이 차원에 붙어 있다**는 뜻이다.

### 왜 18만 박동인데 능력 비용이 나오나 — **유효 표본은 56이다**

LORO 는 **처음 보는 레코드**에 시험한다. 박동은 레코드 안에서 극도로 상관돼 있으므로
일반화의 유효 표본은 18만이 아니라 **레코드 56개**다. 25차원 선형모델은 18만 박동에는
과적합할 수 없어도 **56명의 레코드 고유 리듬 서명**에는 얼마든지 과적합한다.
`ctx`(±4 박동 문맥)가 가장 크게 다친 것도 이와 맞는다 — 문맥 특징은 **그 레코드의 리듬**에
가장 강하게 묶여 있다.

## 이번에 고치는 것 — 설계 결함 셋

| # | Q4-H 의 결함 | Q4-I 의 수정 |
|---|---|---|
| ① | 영점이 음수인데 문턱을 `max(0,·)` 로 **잘라** 기전 판정을 못 읽었다 | **두 판정을 항상 같이** 낸다(배포·기전) |
| ①′ | 영점을 **교정된 점수**로 재서 교정기 부호 되살림이 섞였다 | 영점을 **비교정(raw)** 으로도 재서 **병기**한다 |
| ② | 팔마다 차원이 다른데 `C=1.0` **고정** — 능력이 통제 안 됐다 | **폴드별 DEV 에서 `C` 를 고른다**(R22) |
| ③ | **차원 대조군이 없다** — 차원 탓인지 내용 탓인지 못 가른다 | `shuf` — 추가 블록을 **레코드 안에서 행 치환**(차원·주변분포 동일, 박동 정렬만 파괴) |

⚠️ **특징 블록은 Q4-H 와 한 글자도 안 바꾼다.** 바꾸는 건 **추정기**뿐이다 — 그래야
「문제는 특징이 아니라 추정기다」를 시험한 게 된다.

## ★★★ 그리고 진짜 축 — Q4-H 가 찾아낸 가장 강한 변수

```
불규칙성(RMSSD/중앙 · AF 대리) ~ 달성률  ρ **−0.6797**   ~AUROC **−0.6827**
이단맥 지수                  ~ 달성률  ρ −0.3792        ~유병률 +0.3561
이단맥 상위 절반 vs 하위 — 달성률 0.7032 vs 0.9116 · AUROC 0.9192 vs 0.9647
```

**퀘스트 46 전체에서 나온 단일 설명변수 중 가장 강하다.** 심방세동에서는 모든 RR 이
불규칙하므로 **「이르다」의 기준선 자체가 무너진다** — SVDB 는 심방세동·심방이단맥·조동을
포함한다(PhysioNet).

### 그래서 넣는 네 번째 수정 — **기록별 특징 표준화**

특징을 **그 레코드 자신의 평균·표준편차로 z-화** 한다. 선형모델의 점수는

```
score_r = Σ_j w_j (x_j − μ_jr)/σ_jr  =  Σ_j (w_j/σ_jr) x_j − c_r
```

가 되어 **레코드마다 가중치가 σ_jr 로 자동 조정**된다. AF 레코드는 RR 변동 특징의 σ 가
크므로 그 축의 가중치가 자동으로 내려간다. 레코드 내 순위를 **실제로 바꾼다**(층② 불가능
정리에 걸리지 않는다 — 상수 이동이 아니다).
⚠️ 추론 때 **레코드 전체**가 필요하지만, 우리 처방(환자별 예산 배분)은 **이미** 레코드
전체를 쓴다. 배포 가정이 늘지 않는다.

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **M0** | 코호트 · Platt 기울기 | 구성. 깨지면 **중단** |
| **M1 ★★★** | Q4-H 재현 + **영점 대비 재독** | 관문 아님(정정) |
| **M2 ★★★ 주 관문** | `both` vs `shuf` — **차원 동일**. 내용이 신호를 갖는가 | 측정된 영점 상단 초과 |
| **M3 ★★** | `C` 를 DEV 에서 고르면 `both_t − base_t` 가 뒤집히는가 | 배포·기전 **두 판정** |
| **M4 ★★★** | **기록별 표준화** `both_rz − both` · `base_rz − base` | 배포·기전 두 판정 |
| **M5 ★★** | **불규칙성 축** — 효과가 AF 대리 상·하위에서 다른가 | 관문 아님(사전등록 방향) |
| **M6** | DEV 선택 `sel`(추가 3개만) · 단변량 순위 | 관문 아님 |
| **M7** | 필요표본 · 검산표 | R38 ⑦ · R39 ⑤ · R41 ② |

⚠️ **새 데이터 0** · **GPU 0** — `svdb_data5.npz` 의 `pre_rr`/`post_rr` 파생 + sklearn CPU.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
RHY_K = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25
DEV_EVERY = 4
MAIN_K = 300
CTX = 4                        # Q4-H 와 동일 — 특징은 안 바꾼다
MAX_NEG_SLOPE = 0.10
MIN_AUC_SLOPE = 0.55         # ★ 이보다 판별력이 낮은 팔은 기울기 부호 검사가 무의미
C_GRID = (0.1, 1.0, 10.0)      # ★ 사전 고정. 1.0 을 포함해 Q4-H 를 재현할 수 있게
K_SEL = 3                      # ★ 사전 고정 — DEV 에서 고르는 추가 열 개수
NB_BOOT  = 400 if SMOKE else 2000
N_PERM_A = 2   if SMOKE else 4     # 고정 C 영점(6팔 공유)
N_PERM_B = 1   if SMOKE else 3     # 튜닝 C 영점(2팔)

ARMS_FIX  = ("base", "both", "shuf", "base_rz", "both_rz")
ARMS_NULL = ARMS_FIX + ("sel",)
ARMS_TUNE = ("base_t", "both_t")
CONTRASTS = (("both-base",    "base",    "both"),
             ("both-shuf",    "shuf",    "both"),
             ("both_rz-both", "both",    "both_rz"),
             ("base_rz-base", "base",    "base_rz"),
             ("sel-base",     "base",    "sel"))
MAIN_CT = "both-shuf"
READ_ORDER = ("M0", "M1", "M2", "M3", "M4", "M5", "M6", "M7")
SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(
    n_ok=56, mean_prev=0.0837,
    q4h=dict(auc={"base": 0.9420, "comp": 0.9375, "ctx": 0.9285, "both": 0.9278},
             dims={"base": 9, "comp": 14, "ctx": 20, "both": 25},
             d_comp=(-0.0044, -0.0072, -0.0022), d_ctx=(-0.0134, -0.0201, -0.0075),
             d_both=(-0.0142, -0.0214, -0.0078), null=-0.0427, thr=0.0,
             const_mae=(0.0690, 0.0519, 0.0905), reg=0.0591, mean_p=0.0724, em=0.1790,
             rho_irr_ach=-0.6797, rho_irr_auc=-0.6827, rho_big_ach=-0.3792,
             big_hi=(0.7032, 0.9192, 0.1140), big_lo=(0.9116, 0.9647, 0.0534),
             sens300=0.7459, ach=0.8074, ppv300=0.3620),
    lit=[("de Chazal 2004 (IEEE TBME · DS2)", 0.759, 0.385),
         ("Llamedo & Martinez 2011 (IEEE TBME)", 0.77, 0.39)])

RULE_CHECK = {
    "R11 매크로":       "환자 단위 · 상한과 함께 읽는다",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "LORO — `C` 선택도 **held-out 을 뺀 DEV** 에서만",
    "R26 / R38 ②":      "★★★ 영점을 재고 **자르지 않고** 읽는다 — Q4-H 가 여기서 틀렸다",
    "R29 ② 분기 금지":   "M0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ② 문턱 금지":  "`C_GRID`·`K_SEL`·`CTX`·`MAIN_K` 를 **사전 고정**",
    "R35 ① 자 먼저":    "★★ **차원 대조군 `shuf` 가 자다** — 차원과 내용을 가른다",
    "R38 ⑦ 요약 정합":  "★★★ Q4-H 의 판정을 **철회가 아니라 재독**한다(두 판정 병기)",
    "R39 ① 대안설명":   "차원 단조 + 영점 −0.0427 → **능력 비용** 가설",
    "R40 ① λ ≠ 타당성":  "`C` 를 DEV 로 고르는 건 문턱 조작이 아니다(라벨 누출 없음)",
    "R26 ★ 영점 출처":  "★★★ 영점을 **비교정 점수**로 읽는다 — 교정기가 실제 DEV 라벨로 부호를 되살린다",
    "R36 ⑤ 자기검사":    "★ 기울기 검사는 **팔별**로 — `_rz` 는 레코드 상수를 지워 판별력 없으면 부호가 무작위다",
    "R41 ② 0 근처":     "효과가 0 근처면 필요표본은 해석 불가",
}

CONFIG = dict(
    exp="quest46_q4i_capacity_control", quest="ailab-2026-0046", step="capacity-control",
    parent_exp=["quest46_q4h_rhythm_context"],
    purpose=("★★★ **Q4-H 의 판정을 다시 읽는 런**이다. Q4-H 는 「교과서적 특징이 오히려 "
             "해롭다(L3 ❌)」로 끝났지만, 그 판정을 만든 건 **문턱을 `max(0, 영점상단)` 으로 "
             "자른 것**이었다. 영점(라벨 치환)이 **−0.0427** 이었다 — 즉 **차원을 16개 늘리는 "
             "행위 자체의 값이 −0.0427** 이고, 관측된 −0.0142 는 그 바닥보다 **+0.0285 위**다. "
             "그리고 성능이 특징 내용과 무관하게 **차원에 단조**(9→14→20→25 에서 "
             "0.9420→0.9375→0.9285→0.9278)로 나빠졌다. 두 증거 모두 **비용이 차원에 붙어 "
             "있다**고 말한다. LORO 는 처음 보는 **레코드**에 시험하므로 일반화의 유효 표본은 "
             "18만 박동이 아니라 **레코드 56개**이고, 25차원 선형모델은 56명의 **레코드 고유 "
             "리듬 서명**에 얼마든지 과적합한다(가장 크게 다친 게 `ctx` 인 것과 맞는다). "
             "⇒ **문제의 범위는 「교과서적 생리」가 아니라 「내 추정기」다.** 그래서 이 런은 "
             "**특징 블록을 한 글자도 안 바꾸고** 추정기만 고친다: ① 배포·기전 **두 판정 병기** "
             "② **폴드별 DEV 에서 `C` 선택**(능력 통제) ③ **차원 대조군 `shuf`**(추가 블록을 "
             "레코드 안에서 행 치환 — 차원·주변분포 동일, 박동 정렬만 파괴) ④ **기록별 특징 "
             "표준화**(`_rz`) — 점수가 Σ(w_j/σ_jr)x_j 가 되어 레코드마다 가중치가 자동 조정된다. "
             "④ 는 Q4-H 가 찾아낸 **퀘스트 최강 설명변수**(불규칙성~달성률 ρ **−0.6797**)를 "
             "정면으로 겨눈다 — AF 에서는 RR 변동 특징의 σ 가 커서 그 축이 자동으로 눌린다."),
    dataset="SVDB — svdb_data5.npz (새 데이터 0 · GPU 0 · pre_rr/post_rr 파생만)",
    arms_fix=list(ARMS_FIX), arms_tune=list(ARMS_TUNE), main_contrast=MAIN_CT,
    c_grid=list(C_GRID), k_sel=K_SEL, ctx=CTX, main_k=MAIN_K, min_auc_slope=MIN_AUC_SLOPE,
    read_order=READ_ORDER, dev_every=DEV_EVERY, n_boot=NB_BOOT,
    n_perm_a=N_PERM_A, n_perm_b=N_PERM_B, smoke=SMOKE, ref=REF, rule_check=RULE_CHECK,
    predictions={
        "M0": "코호트 + Platt 기울기. 깨지면 **중단**",
        "M1": "★★★ **정정(관문 아님)** — Q4-H 의 `both − base` 를 재현하고 **영점을 교정·"
              "비교정 두 벌로** 재서 병기한다. 교정 영점과 raw 영점의 차가 크면 Q4-H 의 "
              "−0.0427 은 **능력 비용이 아니라 교정기 부호 되살림**이고, 내 재독이 틀린 것이다",
        "M2": "★★★ **주 관문** — `both` vs `shuf`. **차원이 25로 동일**하므로 능력 비용이 "
              "구성으로 상쇄된다. 추가 특징의 **내용**이 박동 수준 신호를 갖는가",
        "M3": "★★ **능력 통제** — 폴드별 DEV 에서 `C` 를 고른 `both_t − base_t`. Q4-H 의 "
              "−0.0142 가 뒤집히는가",
        "M4": "★★★ **기록별 표준화** — `both_rz − both` · `base_rz − base`. 레코드 고유 "
              "척도를 지우면 전이가 살아나는가",
        "M5": "★★ **불규칙성 축(사전등록 방향)** — 기록별 표준화의 이득이 **AF 대리 상위 "
              "절반에서 더 클 것**이다. 그리고 AF 대리 하위 절반만의 코호트 성능을 낸다",
        "M6": "DEV 에서 고른 추가 3열만 넣은 `sel` · 추가 16열의 단변량 순위",
        "M7": "필요표본 · 결론 검산표"},
    caveat=("★★★ **이 런은 Q4-H 를 철회하지 않는다 — 다시 읽는다.** Q4-H 의 수치는 전부 맞다. "
            "확실히 고칠 것은 하나다: 영점이 −0.0427 인데 문턱을 0 으로 잘라 「기전 판정」을 "
            "아예 안 낸 것 — 이번엔 **두 판정을 항상 같이** 낸다. "
            "★★★ **다만 내 재독이 틀릴 수 있다는 걸 이 런이 스스로 시험한다**: 영점을 "
            "**비교정(raw)** 으로도 재서, 음수 영점이 **교정기 부호 되살림 artifact** 인지 "
            "본다(합성에서는 거의 전부가 artifact 였다). 그래서 **주 관문을 영점 해석에 걸지 "
            "않고**, 차원을 맞춘 `shuf` 대조에 건다. "
            "★★ **특징은 안 바꾼다** — 바꾸면 「추정기가 문제였다」를 시험한 게 아니게 된다. "
            "★★ **`_rz` 는 추론 때 레코드 전체가 필요하다** — 다만 우리 처방(환자별 예산 "
            "배분)은 **이미** 레코드 전체를 쓰므로 배포 가정이 늘지 않는다. "
            "★ **GPU 불필요** — sklearn CPU 로 끝난다. 딥러닝 팔은 이 런에 없다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4i_capacity_control", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-I — 문제는 특징이 아니라 추정기다: 능력·차원 통제**")
run.log(f"  ★★★ Q4-H 재독 — 관측 {REF['q4h']['d_both'][0]:+.4f} · 영점 "
        f"{REF['q4h']['null']:+.4f} ⇒ **영점 위 +{REF['q4h']['d_both'][0]-REF['q4h']['null']:.4f}**")
run.log(f"  ★★★ 주 관문 = `both` vs `shuf` (**차원 25 동일**) — 내용이 신호를 갖는가")
run.log(f"  ★★★ 진짜 축 = **불규칙성(AF 대리)** — Q4-H 실측 ρ(불규칙성, 달성률) "
        f"{REF['q4h']['rho_irr_ach']:+.4f} (퀘스트 최강 설명변수)")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · "
            f"N_PERM_A={N_PERM_A} · N_PERM_B={N_PERM_B})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【M-0】 코호트 · 특징(Q4-H 동일) · ★ 변형 셋(shuf · _rz · 튜닝)
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【M-0】 코호트 · 특징은 **Q4-H 와 동일** · 바꾸는 건 **추정기**뿐")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))
IDX_ALL = {int(r): np.where(RID == r)[0] for r in RS}

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
BASE12 = local_base(12)

F_BASE = np.nan_to_num(np.c_[_med - pre,
                             np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                             post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                             np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                       nan=0.0, posinf=0.0, neginf=0.0)
COMP = (pre + post) / (2.0 * BASE12 + 1e-9)
F_COMP = np.nan_to_num(np.c_[COMP, COMP - 1.0, np.abs(COMP - 1.0),
                             post / (BASE12 + 1e-9), pre / (BASE12 + 1e-9)],
                       nan=0.0, posinf=0.0, neginf=0.0)
def ctx_feats():
    rel = pre / (BASE12 + 1e-9); cols = []
    s = pd.Series(rel); grp = s.groupby(pd.Series(RID))
    for lag in range(1, CTX + 1):
        cols.append(np.nan_to_num(grp.shift(lag).to_numpy(), nan=1.0))
        cols.append(np.nan_to_num(grp.shift(-lag).to_numpy(), nan=1.0))
    short = (rel < 0.90).astype(float)
    sg = pd.Series(short).groupby(pd.Series(RID))
    cols.append(np.nan_to_num(
        sg.transform(lambda x: x.rolling(2 * CTX + 1, center=True, min_periods=1).mean())
        .to_numpy(), nan=0.0))
    cols.append(np.abs(np.nan_to_num(grp.shift(1).to_numpy(), nan=1.0) - rel))
    cols.append(np.abs(np.nan_to_num(grp.shift(-1).to_numpy(), nan=1.0) - rel))
    return np.nan_to_num(np.column_stack(cols), nan=0.0, posinf=0.0, neginf=0.0)
F_CTX = ctx_feats()
ADDED = np.c_[F_COMP, F_CTX]                      # 16열 — Q4-H 의 comp+ctx 그대로

# ── ★★★ 차원 대조군: 추가 블록만 **레코드 안에서 행 단위 공동 치환**
_rs = np.random.RandomState(SEED0 + 7)
ADD_SH = ADDED.copy()
for r in RS:
    ii = IDX_ALL[int(r)]
    ADD_SH[ii] = ADDED[ii][_rs.permutation(len(ii))]
_moved = float(np.mean(np.any(np.abs(ADD_SH - ADDED) > 1e-12, axis=1)))
if _moved < 0.5:
    raise AssetError(f"shuf 가 거의 항등이다({_moved:.3f}) — 대조군 무효(R35 ①)")

# ── ★★★ 기록별 표준화
def rec_z(X):
    Z = np.array(X, float)
    for r in RS:
        ii = IDX_ALL[int(r)]
        mu = X[ii].mean(0); sd = X[ii].std(0) + 1e-9
        Z[ii] = (X[ii] - mu) / sd
    return np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)

FEAT = {"base": F_BASE, "both": np.c_[F_BASE, ADDED], "shuf": np.c_[F_BASE, ADD_SH]}
FEAT["base_rz"] = rec_z(FEAT["base"]); FEAT["both_rz"] = rec_z(FEAT["both"])
FEAT["base_t"] = FEAT["base"]; FEAT["both_t"] = FEAT["both"]
if FEAT["both"].shape[1] != FEAT["shuf"].shape[1]:
    raise AssetError("차원 대조군의 차원이 다르다 — 통제 실패")
run.log(f"  특징 차원 — base {FEAT['base'].shape[1]} · both {FEAT['both'].shape[1]} · "
        f"shuf {FEAT['shuf'].shape[1]}(**both 와 동일**) · 추가 블록 {ADDED.shape[1]}")
run.log(f"  ★ `shuf` 는 추가 16열을 **레코드 안에서 공동 행치환** — 행 {_moved:.1%} 가 움직였다")
run.log(f"  ★ `_rz` 는 특징을 **그 레코드 자신의 μ·σ 로 z-화** — 레코드마다 가중치 자동 조정")

IDXS = {int(r): IDX_ALL[int(r)] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
NS_ = {r: int(TT_[IDXS[r]].sum()) for r in REC_OK}
NRE = len(REC_OK); MEAN_PREV = float(np.mean([BURD[r] for r in REC_OK]))
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}** · 평균 유병률 {MEAN_PREV:.4f}")

SLOPES, SLOPE_BY, _CUR = [], {}, [None]
def make_cal(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(np.asarray(s).reshape(-1, 1),
                                                       np.asarray(y).astype(int))
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    SLOPES.append(a)
    if _CUR[0] is not None: SLOPE_BY.setdefault(_CUR[0], []).append(a)
    return lambda v: a * np.asarray(v, float) + b

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def fit_fold(X, held, y_override, cgrid):
    tr_r, dv_r = split_rest(held)
    tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
    te = IDXS[held]
    Ftr = X[tr]; fmu = Ftr.mean(0); fsd = Ftr.std(0) + 1e-9
    ytr = TT_[tr].astype(int) if y_override is None else np.asarray(y_override[held], int)
    best = None
    for C_ in cgrid:
        lr = LogisticRegression(max_iter=3000, C=C_).fit((Ftr - fmu) / fsd, ytr)
        if len(cgrid) == 1:
            best = (0.0, lr, C_); break
        s_ = lr.decision_function((X[dv] - fmu) / fsd)
        off, a_ = 0, []
        for r in dv_r:                       # ★ DEV 는 held-out 을 뺀 레코드다(R22)
            n_ = len(IDXS[r]); seg = s_[off:off + n_]; off += n_
            yy = TT_[IDXS[r]].astype(int)
            if 0 < yy.sum() < n_: a_.append(roc_auc_score(yy, seg))
        m_ = float(np.mean(a_)) if a_ else -np.inf
        if best is None or m_ > best[0]: best = (m_, lr, C_)
    _, lr, C_ = best
    f = lambda ii: lr.decision_function((X[ii] - fmu) / fsd)
    cl = make_cal(f(dv), TT_[dv])
    raw = f(te)
    return te, cl(raw), raw, C_

CHOSEN_C, SEL_PICKS = {}, {}
def loro(X, y_override=None, cgrid=(1.0,), tag=None):
    out = np.full(len(K), np.nan); raw = np.full(len(K), np.nan)
    picks = []; _CUR[0] = tag
    for held in REC_OK:
        te, v, rv, C_ = fit_fold(X, held, y_override, cgrid)
        out[te] = v; raw[te] = rv; picks.append(C_)
    if tag is not None: CHOSEN_C[tag] = picks
    return out, raw

def pick_cols(dv_r, kpick):
    sc = []
    for j in range(ADDED.shape[1]):
        a_ = []
        for r in dv_r:
            ii = IDXS[r]; yy = TT_[ii].astype(int); v = ADDED[ii, j]
            if 0 < yy.sum() < len(ii) and np.std(v) > 0:
                a_.append(abs(roc_auc_score(yy, v) - 0.5))
        sc.append(float(np.mean(a_)) if a_ else 0.0)
    return [int(j) for j in np.argsort(sc)[::-1][:kpick]]

def loro_sel(kpick=K_SEL, y_override=None, tag=None):
    out = np.full(len(K), np.nan); raw = np.full(len(K), np.nan)
    picks = []; _CUR[0] = tag
    for held in REC_OK:
        _, dv_r = split_rest(held)
        top = pick_cols(dv_r, kpick); picks.append(top)
        te, v, rv, _ = fit_fold(np.c_[F_BASE, ADDED[:, top]], held, y_override, (1.0,))
        out[te] = v; raw[te] = rv
    if tag is not None: SEL_PICKS[tag] = picks
    return out, raw

def run_arm(a, y_override=None, tag=None):
    if a == "sel":  return loro_sel(K_SEL, y_override, tag)
    return loro(FEAT[a], y_override, C_GRID if a.endswith("_t") else (1.0,), tag)

def per_auc(L):
    return {r: float(roc_auc_score(TT_[IDXS[r]].astype(int), L[IDXS[r]])) for r in REC_OK}
def per_ap(L):
    return {r: float(average_precision_score(TT_[IDXS[r]].astype(int), L[IDXS[r]]))
            for r in REC_OK}
def at_k(L, r, k):
    idx = IDXS[r]; sc = L[idx]; yy = TT_[idx]
    k = int(min(max(1, k), len(idx)))
    fl = sc >= np.partition(sc, -k)[-k]
    tp = int((fl & yy).sum()); ceil = min(1.0, k / max(1, NS_[r]))
    return dict(sens=tp / max(1, NS_[r]), ppv=tp / max(1, int(fl.sum())), ceil=ceil,
                ach=(tp / max(1, NS_[r])) / ceil if ceil > 0 else np.nan)
CONFIG["cohort"] = dict(n_ok=NRE, mean_prev=MEAN_PREV, moved=_moved,
                        dims={a: int(FEAT[a].shape[1]) for a in FEAT})
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【M-A】 실행 · M0 · ★★★ M1 Q4-H 재독
run.log("\n" + "=" * 100)
run.log("【M-A】 실행 · M0 · ★★★ M1 — **Q4-H 를 영점 대비로 다시 읽는다**")
run.log("=" * 100)
T0 = time.time()
ALL_ARMS = ARMS_NULL + ARMS_TUNE
L, LRAW = {}, {}
for a in ALL_ARMS:
    L[a], LRAW[a] = run_arm(a, None, tag=a)
    run.log(f"  ({time.time()-T0:>5.0f}초) {a} 완료"
            + (f" · C 중앙 {np.median(CHOSEN_C[a]):g}" if a.endswith("_t") else ""))
AUC = {a: per_auc(L[a]) for a in ALL_ARMS}
AUC_RAW = {a: per_auc(LRAW[a]) for a in ALL_ARMS}
AP  = {a: per_ap(L[a])  for a in ALL_ARMS}
R300 = {a: {r: at_k(L[a], r, MAIN_K) for r in REC_OK} for a in ALL_ARMS}

# ★ Q4-I 수정 — 기울기 검사를 **팔별**로 한다. `_rz` 는 레코드 수준 상수를 지우므로
#   판별력이 없는 조건에서 부호가 무작위가 된다 → **판별력이 있는 팔에서만** 검사한다.
# ★★★ 교정은 **단조 증가**면 레코드 내 AUROC 를 못 바꾼다 — 신호 조건에서 확인한다.
#     (기울기가 음수면 AUROC 가 1−AUROC 로 **뒤집힌다** → 영점 해석에 결정적이다)
CAL_GAP = max(abs(AUC[a][r] - AUC_RAW[a][r]) for a in ALL_ARMS for r in REC_OK)
run.log(f"\n  ★ 교정 전후 레코드별 AUROC 최대차 **{CAL_GAP:.2e}** — 0 이면 Platt 이 "
        f"단조 증가라 AUROC 를 못 바꾼다는 뜻이다")

sl = np.array(SLOPES, float); neg = int((sl <= 0).sum())
bad_arms, skipped = [], []
for a in ALL_ARMS:
    sa = np.array(SLOPE_BY.get(a, []), float)
    mac = float(np.mean(list(AUC[a].values())))
    if len(sa) == 0: continue
    negf = float(np.mean(sa <= 0))
    if mac <= MIN_AUC_SLOPE:
        skipped.append((a, mac)); continue
    if np.median(sa) <= 0 or negf > MAX_NEG_SLOPE:
        bad_arms.append((a, float(np.median(sa)), negf, mac))
    run.log(f"    기울기 `{a:<8}` 중앙 {np.median(sa):+.4f} · 음수 {negf:.1%} · "
            f"매크로 AUROC {mac:.4f}")
for a, mac in skipped:
    run.log(f"    ⚠️ `{a}` 는 매크로 AUROC {mac:.4f} ≤ {MIN_AUC_SLOPE} — **판별력이 없으면 "
            f"기울기 부호는 무의미**하므로 검사를 건너뛴다")
if bad_arms:
    raise AssetError("M0 실패 — 체계적 반전: "
                     + " · ".join(f"{a} 중앙 {m:+.4f} 음수 {f:.1%}" for a, m, f, _ in bad_arms)
                     + " (R29 ②)")
g_("M0", "✅ 지지", f"Platt 기울기 {len(sl)}개(전체 음수 {neg}) · **팔별** 검사 통과 "
                   f"{len(ALL_ARMS) - len(skipped)}/{len(ALL_ARMS)}"
                   + (f" · 건너뜀 {[a for a, _ in skipped]}" if skipped else "")
                   + f" · 차원 대조군 이동 {_moved:.1%}")

run.log(f"\n  {'팔':<10}{'차원':>6}{'매크로 AUROC':>15}{'매크로 PR-AUC':>16}"
        f"{'민감도@300':>13}{'달성률':>10}")
for a in ALL_ARMS:
    d_ = FEAT[a].shape[1] if a in FEAT else FEAT["base"].shape[1] + K_SEL
    run.log(f"  {a:<10}{d_:>6}{np.mean(list(AUC[a].values())):>15.4f}"
            f"{np.mean(list(AP[a].values())):>16.4f}"
            f"{np.mean([R300[a][r]['sens'] for r in REC_OK]):>13.4f}"
            f"{np.mean([R300[a][r]['ach'] for r in REC_OK]):>10.4f}")
run.log(f"  (Q4-H 앵커 — base {REF['q4h']['auc']['base']} · both {REF['q4h']['auc']['both']} · "
        f"민감도@300 {REF['q4h']['sens300']} · 달성률 {REF['q4h']['ach']})")
for a in ARMS_TUNE:
    cc = CHOSEN_C[a]
    run.log(f"  ★ `{a}` 가 폴드별로 고른 C — " + " · ".join(
        f"{c}:{int(np.sum(np.array(cc) == c))}" for c in C_GRID))

# ── ★★★ 영점 A (고정 C · 6팔 공유 · 같은 치환을 모든 팔에 준다)
run.log(f"\n  ★★ **영점 A** — 학습 라벨 치환 뒤 같은 대비 (reps={N_PERM_A} · 6팔 공유)")
run.log(f"    ★★★ 영점은 **비교정(raw) 점수**로 읽는다 — 교정기는 **실제 DEV 라벨**로 "
        f"적합되므로, 학습 신호가 없을 때 **부호를 되살려** 영점을 0.5 위로 밀어올린다. "
        f"그 되살림의 성공률이 차원에 따라 다르면 **영점이 능력 비용처럼 보인다**")
NUL = {c[0]: {r: [] for r in REC_OK} for c in CONTRASTS}
NUL_CAL = {c[0]: {r: [] for r in REC_OK} for c in CONTRASTS}
NAUC = {a: [] for a in ARMS_NULL}; NAUC_CAL = {a: [] for a in ARMS_NULL}
for s_ in range(N_PERM_A):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    RES = {a: run_arm(a, yov) for a in ARMS_NULL}
    An  = {a: per_auc(RES[a][1]) for a in ARMS_NULL}      # ★ raw
    Anc = {a: per_auc(RES[a][0]) for a in ARMS_NULL}      #   교정
    for a in ARMS_NULL:
        NAUC[a].append(float(np.mean(list(An[a].values()))))
        NAUC_CAL[a].append(float(np.mean(list(Anc[a].values()))))
    for nm, x_, y_ in CONTRASTS:
        for r in REC_OK:
            NUL[nm][r].append(An[y_][r] - An[x_][r])
            NUL_CAL[nm][r].append(Anc[y_][r] - Anc[x_][r])
    run.log(f"    ({time.time()-T0:>5.0f}초) 영점 A rep {s_+1}/{N_PERM_A}")
NSTAT, NSTAT_CAL = {}, {}
for nm, x_, y_ in CONTRASTS:
    NSTAT[nm] = boot_mean([float(np.mean(v)) for v in NUL[nm].values()],
                          SEED0 + 61 + len(nm), NB_BOOT)
    NSTAT_CAL[nm] = boot_mean([float(np.mean(v)) for v in NUL_CAL[nm].values()],
                              SEED0 + 71 + len(nm), NB_BOOT)
run.log(f"\n    {'팔':<10}{'영점 매크로 AUROC(raw)':>24}{'(교정)':>12}{'차':>10}")
for a in ARMS_NULL:
    run.log(f"    {a:<10}{np.mean(NAUC[a]):>24.4f}{np.mean(NAUC_CAL[a]):>12.4f}"
            f"{np.mean(NAUC_CAL[a]) - np.mean(NAUC[a]):>+10.4f}")
run.log(f"    ▸ raw 가 0.5 근처인데 교정이 그걸 밀어올렸다면, **Q4-H 의 영점 −0.0427 은 "
        f"능력 비용이 아니라 교정기 부호 되살림의 차원 의존성**이다")
run.log(f"\n    {'대비':<16}{'영점(raw)':>14}{'영점(교정)':>14}{'차':>10}")
for nm, x_, y_ in CONTRASTS:
    run.log(f"    {nm:<16}{NSTAT[nm][0]:>+14.4f}{NSTAT_CAL[nm][0]:>+14.4f}"
            f"{NSTAT_CAL[nm][0] - NSTAT[nm][0]:>+10.4f}")

def two_verdicts(nm, obs):
    """★★★ Q4-H 가 놓친 것 — **배포 판정(절대)** 과 **기전 판정(영점 대비)** 을 같이 낸다"""
    nhi = NSTAT[nm][2]
    thr_dep = max(0.0, nhi) if np.isfinite(nhi) else float("nan")
    return (decide(obs["lo"], obs["hi"], thr_dep, ">"), thr_dep,
            decide(obs["lo"], obs["hi"], nhi, ">"), nhi)

OBS = {}
for nm, x_, y_ in CONTRASTS:
    m_, lo_, hi_, n_ = boot_pair([AUC[x_][r] for r in REC_OK], [AUC[y_][r] for r in REC_OK],
                                 SEED0 + 81 + len(nm), NB_BOOT)
    OBS[nm] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))

run.log(f"\n  ★★★ **M1 — Q4-H 재독**")
o = OBS["both-base"]; nb_ = NSTAT["both-base"]
run.log(f"    관측 `both − base` **{o['mean']:+.4f}** [{o['lo']:+.4f}, {o['hi']:+.4f}]  "
        f"(Q4-H {REF['q4h']['d_both'][0]:+.4f})")
run.log(f"    영점(raw)       **{nb_[0]:+.4f}** [{nb_[1]:+.4f}, {nb_[2]:+.4f}] · "
        f"영점(교정) {NSTAT_CAL['both-base'][0]:+.4f}  (Q4-H 는 **교정 점수**로 쟀다 — "
        f"{REF['q4h']['null']:+.4f})")
run.log(f"    ⇒ **영점 위 신호 {o['mean'] - nb_[0]:+.4f}** — 차원 16개를 늘리는 행위 자체의 "
        f"값이 {nb_[0]:+.4f} 이고, 추가 특징은 그 바닥보다 이만큼 위다")
vd, td, vm, tm = two_verdicts("both-base", o)
run.log(f"    배포 판정(문턱 {td:+.4f}) {vd}   ·   기전 판정(문턱 {tm:+.4f}) {vm}")
run.log(f"    ★ Q4-H 는 **배포 판정만** 보고했다 — 그래서 「특징이 해롭다」로 읽혔다")
_d_null = NSTAT_CAL["both-base"][0] - NSTAT["both-base"][0]
run.log(f"    ★★★ **영점의 출처** — 교정 영점 − raw 영점 = {_d_null:+.4f}. "
        + ("이만큼은 **교정기 부호 되살림**이 만든 것이고 능력 비용이 아니다. "
           "⇒ Q4-H 의 −0.0427 을 그대로 「능력 비용」으로 읽으면 안 된다"
           if abs(_d_null) > 0.005 else
           "거의 0 이다 ⇒ Q4-H 의 영점은 **진짜 능력 비용**이었다"))
g_("M1", "(관문 아님)",
   f"`both − base` {o['mean']:+.4f} · 영점 {nb_[0]:+.4f} · **영점 위 "
   f"{o['mean'] - nb_[0]:+.4f}** — 배포 {vd} / 기전 {vm}")
CONFIG["M0"] = dict(slope_med=float(np.median(sl)), n_neg=neg, moved=_moved,
                    per_arm={a: dict(med=float(np.median(SLOPE_BY[a])),
                                     neg=float(np.mean(np.array(SLOPE_BY[a]) <= 0)))
                             for a in SLOPE_BY},
                    skipped=[a for a, _ in skipped])
CONFIG["M1"] = dict(obs=OBS["both-base"], null=dict(mean=nb_[0], lo=nb_[1], hi=nb_[2]),
                    null_cal={nm: [float(v) for v in NSTAT_CAL[nm][:3]] for nm in NSTAT_CAL},
                    null_auc={a: float(np.mean(NAUC[a])) for a in ARMS_NULL},
                    null_auc_cal={a: float(np.mean(NAUC_CAL[a])) for a in ARMS_NULL},
                    cal_gap=float(CAL_GAP),
                    null_source=float(NSTAT_CAL["both-base"][0] - NSTAT["both-base"][0]),
                    above_null=float(o["mean"] - nb_[0]), verdict_deploy=vd, verdict_mech=vm,
                    macro_auc={a: float(np.mean(list(AUC[a].values()))) for a in ALL_ARMS},
                    sens300={a: float(np.mean([R300[a][r]["sens"] for r in REC_OK]))
                             for a in ALL_ARMS},
                    ach={a: float(np.mean([R300[a][r]["ach"] for r in REC_OK]))
                         for a in ALL_ARMS},
                    chosen_c={a: [float(c) for c in CHOSEN_C[a]] for a in ARMS_TUNE})
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【M-B】 ★★★ M2 차원 대조 · ★★ M3 능력 통제
run.log("\n" + "=" * 100)
run.log("【M-B】 ★★★ M2 — **차원을 25로 맞추고** 내용만 묻는다 · M3 — `C` 를 DEV 에서 고른다")
run.log("=" * 100)
run.log(f"  {'대비':<16}{'Δ 매크로 AUROC':>26}{'영점':>12}{'배포':>10}{'기전':>10}")
TAB = {}
for nm, x_, y_ in CONTRASTS:
    o = OBS[nm]; vd, td, vm, tm = two_verdicts(nm, o)
    TAB[nm] = dict(obs=o, null=dict(mean=NSTAT[nm][0], lo=NSTAT[nm][1], hi=NSTAT[nm][2]),
                   thr_deploy=float(td), thr_mech=float(tm), v_deploy=vd, v_mech=vm,
                   above_null=float(o["mean"] - NSTAT[nm][0]))
    run.log(f"  {nm:<16}{o['mean']:>+10.4f} [{o['lo']:+.4f},{o['hi']:+.4f}]"
            f"{NSTAT[nm][0]:>+12.4f}{vd:>10}{vm:>10}")

om = OBS[MAIN_CT]; vd, td, vm, tm = two_verdicts(MAIN_CT, om)
run.log(f"\n  ★★★ **M2 주 관문** — `{MAIN_CT}` 는 두 팔의 **차원이 {FEAT['both'].shape[1]} 로 "
        f"동일**하므로 능력 비용이 **구성으로 상쇄**된다")
run.log(f"    Δ **{om['mean']:+.4f}** [{om['lo']:+.4f}, {om['hi']:+.4f}] · 영점 "
        f"{NSTAT[MAIN_CT][0]:+.4f} [{NSTAT[MAIN_CT][1]:+.4f}, {NSTAT[MAIN_CT][2]:+.4f}] · "
        f"MDE {om['mde']:.4f}")
g_("M2", vd,
   f"`both` vs `shuf`(차원 동일) Δ {om['mean']:+.4f} [{om['lo']:+.4f}, {om['hi']:+.4f}] · "
   f"배포문턱 {td:+.4f} {vd} · 기전문턱 {tm:+.4f} {vm} — "
   + ("**추가 특징의 내용이 박동 수준 신호를 갖는다**" if vd.startswith("✅") else
      ("내용이 무작위 정렬과 다르지 않다 — 특징 자체가 빈 것이다" if vd.startswith("❌")
       else "가르지 못했다(R33 ①)")))

# ── M3 능력 통제 + 그 영점(튜닝 포함)
run.log(f"\n  ★★ **M3 능력 통제** — 폴드별 DEV 에서 `C ∈ {list(C_GRID)}` 선택(R22)")
m_, lo_, hi_, n_ = boot_pair([AUC["base_t"][r] for r in REC_OK],
                             [AUC["both_t"][r] for r in REC_OK], SEED0 + 91, NB_BOOT)
OBS_T = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))
run.log(f"    ★★ **영점 B** — 튜닝까지 포함한 영점 (reps={N_PERM_B})")
nulT = {r: [] for r in REC_OK}
for s_ in range(N_PERM_B):
    rr = np.random.RandomState(SEED0 + 700 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    pa = per_auc(run_arm("base_t", yov)[1]); pb = per_auc(run_arm("both_t", yov)[1])   # raw
    for r in REC_OK: nulT[r].append(pb[r] - pa[r])
    run.log(f"      ({time.time()-T0:>5.0f}초) 영점 B rep {s_+1}/{N_PERM_B}")
NT = boot_mean([float(np.mean(v)) for v in nulT.values()], SEED0 + 95, NB_BOOT)
td_t = max(0.0, NT[2]) if np.isfinite(NT[2]) else float("nan")
v_dep_t = decide(OBS_T["lo"], OBS_T["hi"], td_t, ">")
v_mec_t = decide(OBS_T["lo"], OBS_T["hi"], NT[2], ">")
run.log(f"    `both_t − base_t` **{OBS_T['mean']:+.4f}** [{OBS_T['lo']:+.4f}, "
        f"{OBS_T['hi']:+.4f}] · 영점 {NT[0]:+.4f} [{NT[1]:+.4f}, {NT[2]:+.4f}]")
run.log(f"    (고정 C 일 때 {OBS['both-base']['mean']:+.4f} → 튜닝하면 "
        f"{OBS_T['mean']:+.4f} · 차 {OBS_T['mean'] - OBS['both-base']['mean']:+.4f})")
g_("M3", v_dep_t,
   f"`both_t − base_t` {OBS_T['mean']:+.4f} [{OBS_T['lo']:+.4f}, {OBS_T['hi']:+.4f}] · "
   f"배포문턱 {td_t:+.4f} {v_dep_t} · 기전문턱 {NT[2]:+.4f} {v_mec_t} · "
   f"능력 통제로 {OBS_T['mean'] - OBS['both-base']['mean']:+.4f} 이동")
CONFIG["M2"] = TAB
CONFIG["M3"] = dict(obs=OBS_T, null=dict(mean=NT[0], lo=NT[1], hi=NT[2]),
                    thr_deploy=float(td_t), thr_mech=float(NT[2]),
                    v_deploy=v_dep_t, v_mech=v_mec_t,
                    shift=float(OBS_T["mean"] - OBS["both-base"]["mean"]))
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【M-C】 ★★★ M4 기록별 표준화 · ★★ M5 불규칙성 축
run.log("\n" + "=" * 100)
run.log("【M-C】 ★★★ M4 — 기록별 표준화 · ★★ M5 — **불규칙성(AF 대리) 축**")
run.log("=" * 100)
for nm in ("base_rz-base", "both_rz-both"):
    t = TAB[nm]
    run.log(f"  {nm:<16} Δ **{t['obs']['mean']:+.4f}** [{t['obs']['lo']:+.4f}, "
            f"{t['obs']['hi']:+.4f}] · 영점 {t['null']['mean']:+.4f} · 배포 {t['v_deploy']} · "
            f"기전 {t['v_mech']}")
best_rz = max(("base_rz-base", "both_rz-both"), key=lambda n: TAB[n]["obs"]["mean"])
g_("M4", TAB[best_rz]["v_deploy"],
   f"최선 `{best_rz}` Δ {TAB[best_rz]['obs']['mean']:+.4f} "
   f"[{TAB[best_rz]['obs']['lo']:+.4f}, {TAB[best_rz]['obs']['hi']:+.4f}] · 배포문턱 "
   f"{TAB[best_rz]['thr_deploy']:+.4f} · 기전 {TAB[best_rz]['v_mech']} — "
   + ("**레코드 고유 척도를 지우면 전이가 살아난다**"
      if TAB[best_rz]["v_deploy"].startswith("✅") else
      ("기록별 표준화가 도움이 안 된다" if TAB[best_rz]["v_deploy"].startswith("❌")
       else "가르지 못했다(R33 ①)")))

# ── M5 불규칙성 축 (사전등록: `_rz` 이득은 **불규칙 상위 절반에서 더 클 것**)
BIG, IRR = {}, {}
for r in REC_OK:
    p = pre[IDXS[r]]; med = float(np.median(p)); sgn = np.sign(p - med)
    BIG[r] = float(np.mean(sgn[:-1] * sgn[1:] < 0))
    IRR[r] = float(np.sqrt(np.mean(np.diff(p) ** 2)) / (med + 1e-9))
cut = float(np.median([IRR[r] for r in REC_OK]))
HI = [r for r in REC_OK if IRR[r] >= cut]; LO = [r for r in REC_OK if IRR[r] < cut]
run.log(f"\n  ★★ M5 — 불규칙성 중앙 {cut:.4f} 로 분할: 상위(AF 대리) {len(HI)} · 하위 {len(LO)}")
run.log(f"  (Q4-H 실측 앵커 — ρ(불규칙성, 달성률) {REF['q4h']['rho_irr_ach']:+.4f} · "
        f"ρ(·, AUROC) {REF['q4h']['rho_irr_auc']:+.4f})")
a0 = "base"
ac = np.array([R300[a0][r]["ach"] for r in REC_OK]); au = np.array([AUC[a0][r] for r in REC_OK])
ir = np.array([IRR[r] for r in REC_OK]); bg = np.array([BIG[r] for r in REC_OK])
run.log(f"    본 런 재측 — ρ(불규칙성, 달성률) {np.corrcoef(ir, ac)[0,1]:+.4f} · "
        f"ρ(·, AUROC) {np.corrcoef(ir, au)[0,1]:+.4f} · ρ(이단맥, 달성률) "
        f"{np.corrcoef(bg, ac)[0,1]:+.4f}")
run.log(f"\n  {'대비':<16}{'상위(불규칙)':>22}{'하위(규칙)':>22}{'차':>10}")
STRAT = {}
for nm, x_, y_ in CONTRASTS:
    h = boot_pair([AUC[x_][r] for r in HI], [AUC[y_][r] for r in HI], SEED0 + 111 + len(nm),
                  NB_BOOT)
    l = boot_pair([AUC[x_][r] for r in LO], [AUC[y_][r] for r in LO], SEED0 + 131 + len(nm),
                  NB_BOOT)
    STRAT[nm] = dict(hi=dict(mean=h[0], lo=h[1], hi=h[2]), lo=dict(mean=l[0], lo=l[1], hi=l[2]),
                     diff=float(h[0] - l[0]))
    run.log(f"  {nm:<16}{h[0]:>+8.4f} [{h[1]:+.4f},{h[2]:+.4f}]{l[0]:>+8.4f} "
            f"[{l[1]:+.4f},{l[2]:+.4f}]{h[0]-l[0]:>+10.4f}")
pre_ok = STRAT["both_rz-both"]["diff"] > 0
run.log(f"\n  ▸ **사전등록 방향** — `both_rz−both` 이득이 불규칙 상위에서 더 크다: "
        + ("✅ 맞았다" if pre_ok else "❌ 틀렸다")
        + f" (차 {STRAT['both_rz-both']['diff']:+.4f})")
run.log(f"\n  ▸ **AF 대리 하위 절반만의 코호트**(=규칙 리듬 환자만 골라 배포한다면)")
for a in ("base", "both_rz"):
    run.log(f"    {a:<9} 전체 AUROC {np.mean([AUC[a][r] for r in REC_OK]):.4f} → 하위만 "
            f"{np.mean([AUC[a][r] for r in LO]):.4f} · 달성률 "
            f"{np.mean([R300[a][r]['ach'] for r in REC_OK]):.4f} → "
            f"{np.mean([R300[a][r]['ach'] for r in LO]):.4f} · 민감도@300 "
            f"{np.mean([R300[a][r]['sens'] for r in REC_OK]):.4f} → "
            f"{np.mean([R300[a][r]['sens'] for r in LO]):.4f}")
g_("M5", "(관문 아님)",
   f"ρ(불규칙성, 달성률) {np.corrcoef(ir, ac)[0,1]:+.4f} 재확인 · `both_rz−both` 상위 "
   f"{STRAT['both_rz-both']['hi']['mean']:+.4f} vs 하위 "
   f"{STRAT['both_rz-both']['lo']['mean']:+.4f} · 사전등록 방향 "
   + ("맞음" if pre_ok else "틀림"))
CONFIG["M4"] = dict(best=best_rz, base_rz=TAB["base_rz-base"], both_rz=TAB["both_rz-both"])
CONFIG["M5"] = dict(cut=cut, n_hi=len(HI), n_lo=len(LO), strat=STRAT, pre_ok=bool(pre_ok),
                    rho_irr_ach=float(np.corrcoef(ir, ac)[0, 1]),
                    rho_irr_auc=float(np.corrcoef(ir, au)[0, 1]),
                    rho_big_ach=float(np.corrcoef(bg, ac)[0, 1]),
                    irr={str(r): IRR[r] for r in REC_OK},
                    lo_only={a: dict(auc=float(np.mean([AUC[a][r] for r in LO])),
                                     ach=float(np.mean([R300[a][r]["ach"] for r in LO])),
                                     sens=float(np.mean([R300[a][r]["sens"] for r in LO])))
                             for a in ALL_ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【M-D】 M6 선택·단변량 · 필요표본 · M7 검산표
run.log("\n" + "=" * 100)
run.log("【M-D】 M6 — DEV 선택 `sel` · 단변량 순위 · 필요표본 · M7 검산표")
run.log("=" * 100)
t = TAB["sel-base"]
run.log(f"  `sel`(base {FEAT['base'].shape[1]} + DEV 선택 {K_SEL}열 = "
        f"{FEAT['base'].shape[1] + K_SEL}차원) Δ **{t['obs']['mean']:+.4f}** "
        f"[{t['obs']['lo']:+.4f}, {t['obs']['hi']:+.4f}] · 영점 {t['null']['mean']:+.4f} · "
        f"배포 {t['v_deploy']} · 기전 {t['v_mech']}")
from collections import Counter
cnt = Counter(j for p in SEL_PICKS.get("sel", []) for j in p)
NAMES = (["comp_ratio", "comp-1", "|comp-1|", "post/base", "pre/base"]
         + [f"rel_lag{l}{s}" for l in range(1, CTX + 1) for s in ("−", "+")]
         + ["ectopic_density", "alt_prev", "alt_next"])
run.log(f"  폴드 {len(SEL_PICKS.get('sel', []))}개가 고른 열 상위 — " + " · ".join(
    f"{NAMES[j] if j < len(NAMES) else j}×{c}" for j, c in cnt.most_common(6)))

run.log(f"\n  ▸ **단변량 순위**(기술통계 · 레코드 내 AUROC 의 |·−0.5| 평균)")
uni = []
for j in range(ADDED.shape[1]):
    a_ = []
    for r in REC_OK:
        ii = IDXS[r]; yy = TT_[ii].astype(int); v = ADDED[ii, j]
        if 0 < yy.sum() < len(ii) and np.std(v) > 0:
            a_.append(abs(roc_auc_score(yy, v) - 0.5))
    uni.append(float(np.mean(a_)) if a_ else 0.0)
for j in np.argsort(uni)[::-1][:6]:
    run.log(f"    {NAMES[j] if j < len(NAMES) else j:<18}{uni[j]:.4f}")
b_ = []
for j in range(F_BASE.shape[1]):
    a_ = []
    for r in REC_OK:
        ii = IDXS[r]; yy = TT_[ii].astype(int); v = F_BASE[ii, j]
        if 0 < yy.sum() < len(ii) and np.std(v) > 0:
            a_.append(abs(roc_auc_score(yy, v) - 0.5))
    b_.append(float(np.mean(a_)) if a_ else 0.0)
run.log(f"    (기존 base 최고 {max(b_):.4f} · 추가 블록 최고 {max(uni):.4f})")
g_("M6", "(관문 아님)",
   f"`sel`({FEAT['base'].shape[1] + K_SEL}차원) Δ {t['obs']['mean']:+.4f} · 배포 "
   f"{t['v_deploy']} · 추가 블록 단변량 최고 {max(uni):.4f} vs base 최고 {max(b_):.4f}")

eff = om["mean"] - TAB[MAIN_CT]["thr_deploy"]
n5 = need_super(NRE, om["mde"], eff); n8 = need_super(NRE, om["mde"], eff, True)
bad = (not np.isfinite(eff)) or abs(eff) < om["mde"]
run.log(f"\n  M2 주 관문  효과-문턱 {eff:+.4f} · 반폭 {om['mde']:.4f} · n(50%) {n5:.0f} · "
        f"n(80%) {n8:.0f}  "
        + ("★ **해석 불가**(R41 ②)" if bad else
           ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다")))

run.log("\n  ★ M7 — **결론 검산표**")
CHECK = [
    dict(claim="★★★ Q4-H 의 「특징이 해롭다」는 **읽는 법의 문제**였다",
         num=f"본 런 재현 — `both − base` {OBS['both-base']['mean']:+.4f} · 영점 "
             f"{NSTAT['both-base'][0]:+.4f} ⇒ **영점 위 "
             f"{CONFIG['M1']['above_null']:+.4f}**. Q4-H 는 문턱을 `max(0,·)` 로 잘라 "
             f"기전 판정을 아예 안 냈다",
         assume="영점 = 학습 라벨 치환. **학습 신호만** 없애고 DEV·시험 라벨은 그대로다",
         iffalse="★★★ 그래서 이 런의 주 관문은 **차원을 맞춘** `both` vs `shuf` 다"),
    dict(claim=f"★★★ M2 — 차원 동일 대조에서 내용의 값은 {om['mean']:+.4f} → {VERD['M2']}",
         num=f"두 팔 모두 {FEAT['both'].shape[1]}차원 · `shuf` 는 추가 16열을 레코드 안에서 "
             f"공동 행치환(행 {_moved:.1%} 이동) · 영점 {NSTAT[MAIN_CT][0]:+.4f}",
         assume="레코드 내 치환이므로 **레코드 수준 정보는 보존**된다 — 파괴한 건 박동 정렬뿐",
         iffalse="★★ 이게 ❌ 면 **특징 자체가 빈 것**이고, ✅ 면 문제는 **능력 비용**이다"),
    dict(claim=f"★★ M3 — `C` 를 DEV 에서 고르면 {CONFIG['M3']['shift']:+.4f} 이동",
         num=f"고정 C {OBS['both-base']['mean']:+.4f} → 튜닝 {OBS_T['mean']:+.4f} · "
             f"영점 {NT[0]:+.4f} · 배포 {CONFIG['M3']['v_deploy']}",
         assume="C 선택은 **held-out 을 뺀 DEV 레코드**의 매크로 AUROC 로만(R22)",
         iffalse="★ 능력이 원인이면 튜닝이 격차를 줄여야 한다 — 안 줄면 원인이 다른 데 있다"),
    dict(claim=f"★★★ M4 — 기록별 표준화 최선 `{best_rz}` "
               f"{TAB[best_rz]['obs']['mean']:+.4f} → {VERD['M4']}",
         num=f"`base_rz−base` {TAB['base_rz-base']['obs']['mean']:+.4f} · `both_rz−both` "
             f"{TAB['both_rz-both']['obs']['mean']:+.4f} · 영점 "
             f"{TAB['both_rz-both']['null']['mean']:+.4f}",
         assume="추론 때 **레코드 전체**가 필요하다 — 우리 처방(환자별 예산)은 이미 그렇다",
         iffalse="★★ 층② 불가능 정리에 안 걸린다 — 상수 이동이 아니라 **가중치가 레코드마다 "
                 "바뀐다**(Σ w_j/σ_jr · x_j)"),
    dict(claim=f"★★ M5 — 불규칙성(AF 대리)이 여전히 최강 설명변수다",
         num=f"본 런 ρ(불규칙성, 달성률) {CONFIG['M5']['rho_irr_ach']:+.4f} "
             f"(Q4-H {REF['q4h']['rho_irr_ach']:+.4f}) · 하위 절반만이면 `base` AUROC "
             f"{CONFIG['M5']['lo_only']['base']['auc']:.4f} · 달성률 "
             f"{CONFIG['M5']['lo_only']['base']['ach']:.4f}",
         assume="RMSSD/중앙 은 **AF 대리**지 AF 진단이 아니다 — 리듬 라벨이 없어서 대리를 쓴다",
         iffalse="★ 하위 절반만의 수치가 크게 좋으면 **적응증을 좁히는 것**이 곧 제품 결정이다"),
]
for i, ck in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{ck['claim']}**")
    run.log(f"      근거   {ck['num']}")
    run.log(f"      가정   {ck['assume']}")
    run.log(f"      틀리면 {ck['iffalse']}")
CONFIG["M6"] = dict(sel=TAB["sel-base"], uni_max=float(max(uni)), base_uni_max=float(max(b_)),
                    uni=[float(u) for u in uni],
                    picked=[[int(j) for j in p] for p in SEL_PICKS.get("sel", [])])
CONFIG["need"] = dict(effect=float(eff), half=float(om["mde"]), sup50=float(n5),
                      sup80=float(n8), uninterpretable=bool(bad))
CONFIG["M7"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【M-E】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
xs = np.arange(len(ALL_ARMS))
ax[0].bar(xs, [np.mean(list(AUC[a].values())) for a in ALL_ARMS], color="tab:blue")
ax[0].set_xticks(xs); ax[0].set_xticklabels(ALL_ARMS, fontsize=7, rotation=30)
ax[0].set_ylim(0.85, 1.0); ax[0].set_ylabel("macro AUROC")
ax[0].set_title("arms (features fixed, estimator varies)", fontsize=9)
ax[0].grid(alpha=.3, axis="y")

nm = [c[0] for c in CONTRASTS]
vv = [OBS[n]["mean"] for n in nm]
lo = [OBS[n]["mean"] - OBS[n]["lo"] for n in nm]; hi = [OBS[n]["hi"] - OBS[n]["mean"] for n in nm]
ax[1].errorbar(vv, np.arange(len(nm)), xerr=[lo, hi], fmt="o", capsize=5, color="tab:red")
ax[1].scatter([NSTAT[n][0] for n in nm], np.arange(len(nm)), marker="x", s=45,
              color="tab:gray", label="measured null")
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels(nm, fontsize=7)
ax[1].set_xlabel("macro AUROC delta")
ax[1].set_title("contrast vs measured null", fontsize=9)
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis="x")

ax[2].scatter([IRR[r] for r in REC_OK], [AUC["base"][r] for r in REC_OK], s=30,
              color="tab:orange", label="base")
ax[2].scatter([IRR[r] for r in REC_OK], [AUC["both_rz"][r] for r in REC_OK], s=22,
              color="tab:green", marker="^", label="both_rz")
ax[2].axvline(cut, ls=":", color="tab:gray", lw=1.2)
ax[2].set_xlabel("irregularity (RMSSD / median RR)  ~ AF proxy")
ax[2].set_ylabel("per-record AUROC")
ax[2].set_title("M5 : the irregularity axis", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q4i_capacity_control", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:7]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  ★★★ **M1 재독** — `both − base` {OBS['both-base']['mean']:+.4f} · 영점 "
        f"{NSTAT['both-base'][0]:+.4f} ⇒ **영점 위 {CONFIG['M1']['above_null']:+.4f}** "
        f"(배포 {CONFIG['M1']['verdict_deploy']} / 기전 {CONFIG['M1']['verdict_mech']})")
run.log(f"  ★★★ **M2 주 관문** — 차원 동일 `both` vs `shuf` {om['mean']:+.4f} "
        f"[{om['lo']:+.4f}, {om['hi']:+.4f}] → {VERD['M2']}")
run.log(f"  ★★ **M3 능력 통제** — 튜닝으로 {CONFIG['M3']['shift']:+.4f} 이동 "
        f"({OBS['both-base']['mean']:+.4f} → {OBS_T['mean']:+.4f}) → {VERD['M3']}")
run.log(f"  ★★★ **M4 기록별 표준화** — `base_rz−base` "
        f"{TAB['base_rz-base']['obs']['mean']:+.4f} · `both_rz−both` "
        f"{TAB['both_rz-both']['obs']['mean']:+.4f} → {VERD['M4']}")
run.log(f"  ★★ **M5 불규칙성 축** — ρ(불규칙성, 달성률) {CONFIG['M5']['rho_irr_ach']:+.4f} · "
        f"규칙 절반만이면 `base` AUROC {CONFIG['M5']['lo_only']['base']['auc']:.4f} · 달성률 "
        f"{CONFIG['M5']['lo_only']['base']['ach']:.4f}")
run.log(f"  ▸ 매크로 AUROC — " + " · ".join(
    f"{a} {np.mean(list(AUC[a].values())):.4f}" for a in ALL_ARMS))
run.log(f"  ▸ **GPU 안 썼다** — sklearn CPU. 딥러닝 팔은 이 런에 없다")

run.finish({
    "exp_id": "quest46_q4i_capacity_control",
    "metric": "macro_auroc_both_minus_shuf",
    "value": float(om["mean"]),
    "passed": bool(ok_("M0") and ok_("M2")),
    "summary": ("Q4-H 의 「교과서적 특징이 해롭다」를 **철회가 아니라 재독**한 런. Q4-H 의 "
                "영점은 −0.0427 이었는데 문턱을 `max(0, 영점상단)` 으로 잘라 **기전 판정을 "
                "아예 안 냈다** — 관측 −0.0142 는 그 바닥보다 +0.0285 위였다. 게다가 성능이 "
                "특징 내용과 무관하게 **차원에 단조**로 나빠졌다(9→14→20→25). LORO 의 유효 "
                "표본은 18만 박동이 아니라 **레코드 56개**이므로 25차원 선형모델이 레코드 고유 "
                "리듬 서명에 과적합할 수 있다. ⇒ **범위는 생리가 아니라 추정기다.** 그래서 "
                "**특징을 한 글자도 안 바꾸고** 추정기만 고쳤다: 배포·기전 **두 판정 병기** · "
                "**차원 동일 대조군 `shuf`** · **폴드별 DEV 에서 `C` 선택** · **기록별 특징 "
                "표준화 `_rz`**. `_rz` 는 Q4-H 최강 설명변수(불규칙성~달성률 ρ −0.6797)를 "
                "정면으로 겨눈다 — 점수가 Σ(w_j/σ_jr)x_j 가 되어 AF 레코드에서 변동 축의 "
                "가중치가 자동으로 눌린다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "M0": CONFIG.get("M0", {}),
    "M1": CONFIG.get("M1", {}), "M2": CONFIG.get("M2", {}), "M3": CONFIG.get("M3", {}),
    "M4": CONFIG.get("M4", {}), "M5": CONFIG.get("M5", {}), "M6": CONFIG.get("M6", {}),
    "need": CONFIG.get("need", {}), "M7": CONFIG.get("M7", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4i_capacity_control.ipynb`")
